# Specialist LLM SFT on Kaggle (Qwen3.5 + Unsloth + QLoRA)

**Before you run**
1. Accelerator: **GPU T4 x2** (not P100: Unsloth needs CUDA capability 7.0+).
2. Settings: **Internet = On**.
3. Add-ons > Secrets: add `HF_TOKEN` (a Hugging Face *write* token).
4. Add your dataset (Add Input) containing `train.jsonl`, one JSON object per line:
   `{"prompt": "...", "completion": "..."}`. Update `train_file` below.
5. For long runs use **Save & Run All (Commit)** so it runs headless. An idle open notebook still burns quota.

This notebook trains on ONE GPU. Your pass/fail eval harness lives elsewhere; never put eval tasks in `train.jsonl`.

In [ ]:
import os

GPU_ID = "0"                      # T4 x2 gives GPU 0 and 1; this notebook uses one
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CFG = dict(
    model_name      = "unsloth/Qwen3.5-4B",   # iterate on 4B, switch to unsloth/Qwen3.5-9B for the final run
    load_in_4bit    = True,
    max_seq_len     = 2048,
    lora_r          = 16,
    lora_alpha      = 16,
    lora_dropout    = 0.0,
    epochs          = 2,
    lr              = 2e-4,
    batch_size      = 1,
    grad_accum      = 8,
    max_train_hours = 8.0,            # graceful stop + save before the session limit
    save_steps      = 100,
    train_file      = "/kaggle/input/fastapi-sft/train.jsonl",   # <-- change to your dataset path
    out_dir         = "/kaggle/working/outputs",
    adapter_dir     = "/kaggle/working/adapter",
    hf_repo         = "your-hf-username/qwen35-4b-fastapi-lora", # <-- change
    push_to_hub     = True,
    system_prompt   = "You are an expert FastAPI backend engineer. Use FastAPI, Pydantic v2 and SQLModel. Return complete, runnable code.",
    seed            = 3407,
)
print(CFG["model_name"], "| seq", CFG["max_seq_len"], "| effective batch", CFG["batch_size"] * CFG["grad_accum"])

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo

In [ ]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Hugging Face login OK")
except Exception as e:
    print("HF login failed, pushing to the Hub is disabled:", e)
    CFG["push_to_hub"] = False

In [ ]:
import time
import torch
from unsloth import FastLanguageModel

print("torch", torch.__version__, "|", torch.cuda.get_device_name(0),
      "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

Loader = FastLanguageModel
try:
    model, tokenizer = Loader.from_pretrained(
        model_name=CFG["model_name"],
        max_seq_length=CFG["max_seq_len"],
        load_in_4bit=CFG["load_in_4bit"],
        dtype=None,                    # auto: fp16 on T4 (no bf16 support)
    )
except Exception as e:
    print("FastLanguageModel failed, falling back to FastModel:", e)
    from unsloth import FastModel
    Loader = FastModel
    model, tokenizer = Loader.from_pretrained(
        model_name=CFG["model_name"],
        max_seq_length=CFG["max_seq_len"],
        load_in_4bit=CFG["load_in_4bit"],
        dtype=None,
    )

# Qwen3.5 is multimodal, so "tokenizer" may be a processor. Use the plain text tokenizer.
tok = getattr(tokenizer, "tokenizer", tokenizer)
print("loaded:", CFG["model_name"])

In [ ]:
model = Loader.get_peft_model(
    model,
    r=CFG["lora_r"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=CFG["seed"],
)
model.print_trainable_parameters()

## Data

Training text is built so it matches inference exactly: the prompt part comes from the chat template with
`add_generation_prompt=True` and **thinking disabled**, followed by the completion and `<|im_end|>`.
Loss is computed on the response only. Examples longer than `max_seq_len` are dropped (and counted).

In [ ]:
import json
from datasets import Dataset

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

if os.path.exists(CFG["train_file"]):
    rows = read_jsonl(CFG["train_file"])
    print("loaded", len(rows), "rows from", CFG["train_file"])
else:
    print("!! train file not found. Using a 4-row toy set ONLY to smoke-test the notebook.")
    rows = [
        {"prompt": "Create a FastAPI app with a GET /health endpoint returning {'status': 'ok'}.",
         "completion": "from fastapi import FastAPI\n\napp = FastAPI()\n\n\n@app.get('/health')\ndef health():\n    return {'status': 'ok'}\n"},
        {"prompt": "Write a Pydantic v2 model Item with name (str) and price (float > 0).",
         "completion": "from pydantic import BaseModel, Field\n\n\nclass Item(BaseModel):\n    name: str\n    price: float = Field(gt=0)\n"},
        {"prompt": "Add a POST /items endpoint that accepts an Item and returns it.",
         "completion": "from fastapi import FastAPI\nfrom pydantic import BaseModel, Field\n\napp = FastAPI()\n\n\nclass Item(BaseModel):\n    name: str\n    price: float = Field(gt=0)\n\n\n@app.post('/items')\ndef create_item(item: Item):\n    return item\n"},
        {"prompt": "Write a GET /items/{item_id} endpoint that returns 404 if the id is unknown.",
         "completion": "from fastapi import FastAPI, HTTPException\n\napp = FastAPI()\nDB = {1: {'name': 'pen'}}\n\n\n@app.get('/items/{item_id}')\ndef get_item(item_id: int):\n    if item_id not in DB:\n        raise HTTPException(status_code=404, detail='Item not found')\n    return DB[item_id]\n"},
    ]

def build_text(r):
    msgs = [
        {"role": "system", "content": CFG["system_prompt"]},
        {"role": "user", "content": r["prompt"]},
    ]
    try:
        prompt_text = tok.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    except TypeError:
        prompt_text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return prompt_text + r["completion"].rstrip() + "<|im_end|>\n"

texts, dropped = [], 0
for r in rows:
    t = build_text(r)
    if len(tok(t, add_special_tokens=False)["input_ids"]) > CFG["max_seq_len"]:
        dropped += 1
        continue
    texts.append({"text": t})

train_ds = Dataset.from_list(texts).shuffle(seed=CFG["seed"])
print(f"kept {len(train_ds)} examples, dropped {dropped} (too long)")
print("---- sample ----")
print(train_ds[0]["text"][:600])

In [ ]:
import inspect
from glob import glob
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
from unsloth import is_bfloat16_supported


class TimeLimit(TrainerCallback):
    # Stops training and forces a checkpoint save when the time budget is hit.
    def __init__(self, hours):
        self.t0 = time.time()
        self.limit = hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.t0 > self.limit:
            print(f"time limit reached at step {state.global_step}, saving and stopping")
            control.should_save = True
            control.should_training_stop = True
        return control


use_bf16 = is_bfloat16_supported()
want = dict(
    output_dir=CFG["out_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    num_train_epochs=CFG["epochs"],
    learning_rate=CFG["lr"],
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.01,
    optim="adamw_8bit",
    logging_steps=5,
    save_strategy="steps",
    save_steps=CFG["save_steps"],
    save_total_limit=2,
    fp16=not use_bf16,
    bf16=use_bf16,
    seed=CFG["seed"],
    report_to="none",
    dataset_text_field="text",
    packing=False,
    dataset_num_proc=2,
    push_to_hub=CFG["push_to_hub"],
    hub_model_id=CFG["hf_repo"],
    hub_strategy="every_save",
    hub_private_repo=True,
)
ok = set(inspect.signature(SFTConfig.__init__).parameters)
want["max_length" if "max_length" in ok else "max_seq_length"] = CFG["max_seq_len"]
args = SFTConfig(**{k: v for k, v in want.items() if k in ok})

trainer_kwargs = dict(model=model, train_dataset=train_ds, args=args,
                      callbacks=[TimeLimit(CFG["max_train_hours"])])
try:
    trainer = SFTTrainer(processing_class=tok, **trainer_kwargs)
except TypeError:
    trainer = SFTTrainer(tokenizer=tok, **trainer_kwargs)

# Loss on the assistant response only
try:
    from unsloth.chat_templates import train_on_responses_only
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )
    print("response-only loss: ON")
except Exception as e:
    print("response-only loss unavailable, training on full text:", e)

In [ ]:
resume = len(glob(os.path.join(CFG["out_dir"], "checkpoint-*"))) > 0
print("resuming from local checkpoint" if resume else "fresh run")
stats = trainer.train(resume_from_checkpoint=resume)
print(stats.metrics)
print("peak GPU memory (GB):", round(torch.cuda.max_memory_reserved() / 1e9, 2))

In [ ]:
# Quick sanity check: does the model produce sensible code in the trained format?
Loader.for_inference(model)

msgs = [
    {"role": "system", "content": CFG["system_prompt"]},
    {"role": "user", "content": "Create a FastAPI app with a /health endpoint and a POST /items endpoint using Pydantic v2 models."},
]
try:
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True, enable_thinking=False)
except TypeError:
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

inputs = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
out = model.generate(**inputs, max_new_tokens=512, do_sample=True, temperature=0.2, top_p=0.9)
print(tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

In [ ]:
# Save the LoRA adapter (small) locally and to the Hub
model.save_pretrained(CFG["adapter_dir"])
tok.save_pretrained(CFG["adapter_dir"])
print("saved adapter to", CFG["adapter_dir"])

if CFG["push_to_hub"]:
    try:
        model.push_to_hub(CFG["hf_repo"], private=True)
        tok.push_to_hub(CFG["hf_repo"], private=True)
        print("pushed to", CFG["hf_repo"])
    except Exception as e:
        print("push failed (adapter is still in /kaggle/working/adapter):", e)

## Next steps

1. Run your pass/fail eval harness against this adapter and compare with the untouched base model.
2. If it helps, do round 2: sample from this model, keep only samples that pass tests, retrain on the combined data.
3. For the final release, switch `model_name` to the 9B, re-run, then merge and export (GGUF etc.) in a separate notebook.

**Resume after a session ends:** `/kaggle/working` is cleared between sessions unless you saved the notebook output.
The adapter and checkpoints are pushed to your Hub repo on every save, so you can download them and attach them as an input.